In [ ]:
# ============================================================
# STEP 13B.1 — RESTORE FINAL 10-STATION DATA
# ============================================================

from google.colab import drive

import os
import json
import joblib
import numpy as np
import pandas as pd

drive.mount("/content/drive")

PROJECT_PATH = (
    "/content/drive/MyDrive/Delhi-AQI-Prediction"
)

PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed"
)

MODEL_PATH = os.path.join(
    PROJECT_PATH,
    "models"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

os.makedirs(
    OUTPUT_PATH,
    exist_ok=True
)

os.makedirs(
    TABLE_PATH,
    exist_ok=True
)

model_df = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "delhi_10_station_feature_engineered.parquet"
    )
)

model_df["Datetime"] = pd.to_datetime(
    model_df["Datetime"]
)

model_df["StationId"] = (
    model_df["StationId"]
    .astype(str)
)

model_df = (
    model_df
    .sort_values(
        ["Datetime", "StationId"]
    )
    .reset_index(drop=True)
)

print("=" * 60)
print("FINAL 10-STATION DATA RESTORED")
print("=" * 60)

print(
    "Rows:",
    len(model_df)
)

print(
    "Stations:",
    model_df["StationId"].nunique()
)

print(
    "Start:",
    model_df["Datetime"].min()
)

print(
    "End:",
    model_df["Datetime"].max()
)

In [ ]:
# ============================================================
# STEP 13B.2 — FEATURE CONTRACT
# ============================================================

RAW_FEATURES = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

TARGET = "AQI_t_plus_1"

current_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

lag_features = [
    f"{column}_lag_{lag}h"
    for lag in [1, 2, 12]
    for column in RAW_FEATURES
]

rolling_features = [
    "AQI_rolling_mean_3h",
    "AQI_rolling_std_3h",
    "AQI_rolling_mean_12h",
    "AQI_rolling_std_12h",

    "PM2.5_rolling_mean_3h",
    "PM2.5_rolling_std_3h",
    "PM2.5_rolling_mean_12h",
    "PM2.5_rolling_std_12h",

    "PM10_rolling_mean_3h",
    "PM10_rolling_std_3h",
    "PM10_rolling_mean_12h",
    "PM10_rolling_std_12h",

    "NO2_rolling_mean_3h",
    "NO2_rolling_std_3h",
    "NO2_rolling_mean_12h",
    "NO2_rolling_std_12h"
]

calendar_features = [
    "Hour",
    "Day",
    "Month",
    "Year",
    "DayOfWeek",
    "IsWeekend"
]

event_features = [
    "IsStubbleSeason",
    "IsDiwaliWindow"
]

numeric_features = (
    current_features
    + lag_features
    + rolling_features
    + calendar_features
    + event_features
)

categorical_features = [
    "StationId"
]

model_features = (
    numeric_features
    + categorical_features
)

missing_features = [
    column
    for column in model_features
    if column not in model_df.columns
]

assert not missing_features, (
    f"Missing model features: {missing_features}"
)

print(
    "Total model inputs:",
    len(model_features)
)

In [ ]:
# ============================================================
# STEP 13B.3 — FIXED FINAL TEST PERIOD
# ============================================================

unique_times = np.sort(
    model_df["Datetime"].unique()
)

split_position = int(
    len(unique_times) * 0.80
)

final_test_start = unique_times[
    split_position
]

optimization_df = (
    model_df[
        model_df["Datetime"]
        < final_test_start
    ]
    .copy()
    .reset_index(drop=True)
)

final_test_df = (
    model_df[
        model_df["Datetime"]
        >= final_test_start
    ]
    .copy()
    .reset_index(drop=True)
)

print("=" * 60)
print("OPTIMIZATION / FINAL TEST SPLIT")
print("=" * 60)

print(
    "Optimization data:",
    len(optimization_df)
)

print(
    "Optimization end:",
    optimization_df["Datetime"].max()
)

print(
    "\nFinal test data:",
    len(final_test_df)
)

print(
    "Final test start:",
    final_test_df["Datetime"].min()
)

assert (
    optimization_df["Datetime"].max()
    <
    final_test_df["Datetime"].min()
)

In [ ]:
# ============================================================
# STEP 13B.4 — PREPROCESSING
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_transformer,
                numeric_features
            ),
            (
                "categorical",
                categorical_transformer,
                categorical_features
            )
        ]
    )

print("Preprocessing factory ready.")

In [ ]:
# ============================================================
# STEP 13B.5 — OPTIMIZATION MATRICES
# ============================================================

X_opt = optimization_df[
    model_features
].copy()

y_opt = optimization_df[
    TARGET
].copy()

X_final_test = final_test_df[
    model_features
].copy()

y_final_test = final_test_df[
    TARGET
].copy()

print(
    "Optimization matrix:",
    X_opt.shape
)

print(
    "Final test matrix:",
    X_final_test.shape
)

In [ ]:
# ============================================================
# STEP 13B.6 — METRICS
# ============================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

def prediction_accuracy_10(
    actual,
    predicted
):
    actual = np.asarray(
        actual,
        dtype=float
    )

    predicted = np.asarray(
        predicted,
        dtype=float
    )

    valid = (
        np.isfinite(actual)
        &
        np.isfinite(predicted)
    )

    actual = actual[valid]
    predicted = predicted[valid]

    if len(actual) == 0:
        return np.nan

    return (
        np.mean(
            np.abs(
                actual - predicted
            ) <= 10
        )
        * 100
    )


def evaluate_predictions(
    actual,
    predicted
):
    actual = np.asarray(
        actual,
        dtype=float
    )

    predicted = np.asarray(
        predicted,
        dtype=float
    )

    valid = (
        np.isfinite(actual)
        &
        np.isfinite(predicted)
    )

    actual = actual[valid]
    predicted = predicted[valid]

    return {
        "MAE": mean_absolute_error(
            actual,
            predicted
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                actual,
                predicted
            )
        ),
        "R2": r2_score(
            actual,
            predicted
        ),
        "Accuracy_±10_AQI_%": prediction_accuracy_10(
            actual,
            predicted
        )
    }

In [ ]:
# ============================================================
# STEP 13B.7 — ROLLING TIME VALIDATION
# ============================================================

from sklearn.model_selection import TimeSeriesSplit

# Use timestamps rather than individual station rows
# so the validation boundary is a real time boundary.

optimization_times = np.sort(
    optimization_df["Datetime"].unique()
)

n_splits = 4

time_splitter = TimeSeriesSplit(
    n_splits=n_splits
)

fold_boundaries = []

for fold, (
    train_indices,
    validation_indices
) in enumerate(
    time_splitter.split(
        optimization_times
    ),
    start=1
):

    train_times = optimization_times[
        train_indices
    ]

    validation_times = optimization_times[
        validation_indices
    ]

    fold_boundaries.append(
        {
            "Fold": fold,
            "TrainStart": train_times.min(),
            "TrainEnd": train_times.max(),
            "ValidationStart": validation_times.min(),
            "ValidationEnd": validation_times.max()
        }
    )

fold_boundaries = pd.DataFrame(
    fold_boundaries
)

display(
    fold_boundaries
)

In [ ]:
# ============================================================
# STEP 13B.8 — RF ROLLING EVALUATOR
# ============================================================

from sklearn.ensemble import RandomForestRegressor

def evaluate_rf_config(
    params
):

    fold_results = []

    for fold, (
        train_indices,
        validation_indices
    ) in enumerate(
        time_splitter.split(
            optimization_times
        ),
        start=1
    ):

        train_times = optimization_times[
            train_indices
        ]

        validation_times = optimization_times[
            validation_indices
        ]

        fold_train = optimization_df[
            optimization_df["Datetime"].isin(
                train_times
            )
        ]

        fold_valid = optimization_df[
            optimization_df["Datetime"].isin(
                validation_times
            )
        ]

        X_train_fold = fold_train[
            model_features
        ]

        y_train_fold = fold_train[
            TARGET
        ]

        X_valid_fold = fold_valid[
            model_features
        ]

        y_valid_fold = fold_valid[
            TARGET
        ]

        model = Pipeline(
            steps=[
                (
                    "preprocessor",
                    make_preprocessor()
                ),
                (
                    "model",
                    RandomForestRegressor(
                        **params
                    )
                )
            ]
        )

        model.fit(
            X_train_fold,
            y_train_fold
        )

        predictions = model.predict(
            X_valid_fold
        )

        metrics = evaluate_predictions(
            y_valid_fold,
            predictions
        )

        metrics["Fold"] = fold

        fold_results.append(
            metrics
        )

    fold_df = pd.DataFrame(
        fold_results
    )

    return {
        "MAE": fold_df["MAE"].mean(),
        "RMSE": fold_df["RMSE"].mean(),
        "R2": fold_df["R2"].mean(),
        "Accuracy_±10_AQI_%":
            fold_df[
                "Accuracy_±10_AQI_%"
            ].mean()
    }

In [ ]:
# ============================================================
# STEP 13B.9 — RF HYPERPARAMETER SEARCH
# ============================================================

candidate_configs = [

    {
        "n_estimators": 200,
        "max_depth": 14,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
        "random_state": 42,
        "n_jobs": -1
    },

    {
        "n_estimators": 250,
        "max_depth": 18,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
        "random_state": 42,
        "n_jobs": -1
    },

    {
        "n_estimators": 300,
        "max_depth": 22,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
        "random_state": 42,
        "n_jobs": -1
    },

    {
        "n_estimators": 250,
        "max_depth": 18,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
        "random_state": 42,
        "n_jobs": -1
    },

    {
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
        "random_state": 42,
        "n_jobs": -1
    },

    {
        "n_estimators": 300,
        "max_depth": 22,
        "min_samples_leaf": 10,
        "max_features": 0.7,
        "random_state": 42,
        "n_jobs": -1
    }
]

search_results = []

for config_id, params in enumerate(
    candidate_configs,
    start=1
):

    print(
        f"\nTesting configuration "
        f"{config_id}/{len(candidate_configs)}"
    )

    print(params)

    metrics = evaluate_rf_config(
        params
    )

    search_results.append(
        {
            "ConfigID": config_id,
            **params,
            **metrics
        }
    )

rf_search_results = pd.DataFrame(
    search_results
)

rf_search_results = (
    rf_search_results
    .sort_values(
        "RMSE"
    )
    .reset_index(drop=True)
)

print(
    "\n" + "=" * 80
)

print(
    "RANDOM FOREST HYPERPARAMETER SEARCH"
)

print(
    "=" * 80
)

display(
    rf_search_results[
        [
            "ConfigID",
            "n_estimators",
            "max_depth",
            "min_samples_leaf",
            "max_features",
            "MAE",
            "RMSE",
            "R2",
            "Accuracy_±10_AQI_%"
        ]
    ].round(4)
)

In [ ]:
# ============================================================
# STEP 13B.10 — BEST RF CONFIG
# ============================================================

best_rf_row = (
    rf_search_results
    .sort_values(
        "RMSE"
    )
    .iloc[0]
)

best_config_id = int(
    best_rf_row["ConfigID"]
)

best_rf_params = (
    candidate_configs[
        best_config_id - 1
    ]
)

print("=" * 60)
print("BEST RF CONFIGURATION")
print("=" * 60)

print(
    best_rf_params
)

print(
    "\nRolling CV MAE:",
    round(
        best_rf_row["MAE"],
        4
    )
)

print(
    "Rolling CV RMSE:",
    round(
        best_rf_row["RMSE"],
        4
    )
)

print(
    "Rolling CV R²:",
    round(
        best_rf_row["R2"],
        6
    )
)

print(
    "Rolling CV Accuracy ±10:",
    round(
        best_rf_row[
            "Accuracy_±10_AQI_%"
        ],
        2
    ),
    "%"
)

In [ ]:
# ============================================================
# STEP 13B.11 — TRAIN TUNED RF
# ============================================================

tuned_rf = Pipeline(
    steps=[
        (
            "preprocessor",
            make_preprocessor()
        ),
        (
            "model",
            RandomForestRegressor(
                **best_rf_params
            )
        )
    ]
)

tuned_rf.fit(
    X_opt,
    y_opt
)

print(
    "Tuned Random Forest trained."
)

In [ ]:
# ============================================================
# STEP 13B.12 — FINAL TEST OF TUNED RF
# ============================================================

tuned_rf_prediction = tuned_rf.predict(
    X_final_test
)

tuned_final_metrics = evaluate_predictions(
    y_final_test,
    tuned_rf_prediction
)

print("=" * 70)
print("TUNED RANDOM FOREST — FINAL TEST")
print("=" * 70)

print(
    "MAE:",
    round(
        tuned_final_metrics["MAE"],
        4
    )
)

print(
    "RMSE:",
    round(
        tuned_final_metrics["RMSE"],
        4
    )
)

print(
    "R²:",
    round(
        tuned_final_metrics["R2"],
        6
    )
)

print(
    "Accuracy ±10 AQI:",
    round(
        tuned_final_metrics[
            "Accuracy_±10_AQI_%"
        ],
        2
    ),
    "%"
)

In [ ]:
# ============================================================
# STEP 13B.13 — CHAMPION VS TUNED RF
# ============================================================

current_champion_metrics = {
    "Model": "Current Random Forest",
    "MAE": 3.497741,
    "RMSE": 8.457054,
    "R2": 0.995353,
    "Accuracy_±10_AQI_%": 93.59
}

tuned_metrics_row = {
    "Model": "Tuned Random Forest",
    **{
        key: value
        for key, value
        in tuned_final_metrics.items()
        if key in [
            "MAE",
            "RMSE",
            "R2",
            "Accuracy_±10_AQI_%"
        ]
    }
}

champion_comparison = pd.DataFrame([
    current_champion_metrics,
    tuned_metrics_row
])

champion_comparison[
    "RMSE_Change_vs_Current"
] = (
    champion_comparison["RMSE"]
    - current_champion_metrics["RMSE"]
)

champion_comparison[
    "Accuracy_Change_vs_Current"
] = (
    champion_comparison[
        "Accuracy_±10_AQI_%"
    ]
    -
    current_champion_metrics[
        "Accuracy_±10_AQI_%"
    ]
)

display(
    champion_comparison.round(
        6
    )
)

In [ ]:
# ============================================================
# STEP 13B.14 — MODEL PROMOTION DECISION
# ============================================================

tuned_rmse = (
    tuned_final_metrics["RMSE"]
)

tuned_accuracy = (
    tuned_final_metrics[
        "Accuracy_±10_AQI_%"
    ]
)

current_rmse = (
    current_champion_metrics["RMSE"]
)

current_accuracy = (
    current_champion_metrics[
        "Accuracy_±10_AQI_%"
    ]
)

rmse_improved = (
    tuned_rmse
    <
    current_rmse
)

accuracy_not_materially_worse = (
    tuned_accuracy
    >=
    current_accuracy - 0.50
)

promote_tuned_rf = (
    rmse_improved
    and
    accuracy_not_materially_worse
)

print("=" * 70)
print("RANDOM FOREST PROMOTION DECISION")
print("=" * 70)

print(
    "RMSE improved:",
    rmse_improved
)

print(
    "Accuracy acceptable:",
    accuracy_not_materially_worse
)

print(
    "Promote tuned RF:",
    promote_tuned_rf
)

In [ ]:
# ============================================================
# STEP 13B.15 — SAVE OPTIMIZATION ARTIFACTS
# ============================================================

rf_search_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase13B_random_forest_search_results.csv"
    ),
    index=False
)

champion_comparison.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase13B_champion_vs_tuned_rf.csv"
    ),
    index=False
)

joblib.dump(
    tuned_rf,
    os.path.join(
        MODEL_PATH,
        "phase13B_tuned_random_forest.joblib"
    )
)

print(
    "Optimization artifacts saved."
)

In [ ]:
# ============================================================
# STEP 13B.16 — SAVE PHASE 13B DECISION
# ============================================================

if promote_tuned_rf:

    selected_model_name = (
        "Tuned Random Forest"
    )

    selected_model_file = (
        "phase13B_tuned_random_forest.joblib"
    )

else:

    selected_model_name = (
        "Current Random Forest"
    )

    selected_model_file = (
        "phase12_5_10_station_random_forest.joblib"
    )

phase13B_summary = {
    "phase": (
        "Phase 13B - Random Forest Optimization"
    ),

    "selected_model": selected_model_name,

    "selected_model_file": selected_model_file,

    "tuned_model_promoted": bool(
        promote_tuned_rf
    ),

    "current_champion_rmse": float(
        current_rmse
    ),

    "current_champion_accuracy_10": float(
        current_accuracy
    ),

    "tuned_rmse": float(
        tuned_rmse
    ),

    "tuned_accuracy_10": float(
        tuned_accuracy
    ),

    "next_phase": (
        "Phase 14 - Feature Optimization and Ablation"
    )
}

phase13B_path = os.path.join(
    OUTPUT_PATH,
    "phase13B_summary.json"
)

with open(
    phase13B_path,
    "w"
) as f:
    json.dump(
        phase13B_summary,
        f,
        indent=4
    )

print(
    "Phase 13B summary saved."
)

In [ ]:
# ============================================================
# STEP 13B.17 — FINAL PHASE 13B REPORT
# ============================================================

print("=" * 75)
print("PHASE 13B — RANDOM FOREST OPTIMIZATION")
print("=" * 75)

print(
    "Current champion:",
    selected_model_name
)

print(
    "\nCurrent RF:"
)

print(
    "RMSE:",
    current_rmse
)

print(
    "Accuracy ±10:",
    current_accuracy,
    "%"
)

print(
    "\nTuned RF:"
)

print(
    "RMSE:",
    tuned_rmse
)

print(
    "Accuracy ±10:",
    tuned_accuracy,
    "%"
)

print(
    "\nTuned RF promoted:",
    promote_tuned_rf
)

print(
    "\nNext:",
    "Phase 14 - Feature Optimization and Ablation"
)

print("=" * 75)

In [ ]:
# ============================================================
# PHASE 13B — CHECKPOINT
# ============================================================

import os
import json

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

os.makedirs(
    OUTPUT_PATH,
    exist_ok=True
)

os.makedirs(
    TABLE_PATH,
    exist_ok=True
)

phase13B_checkpoint = {
    "completed_phase": (
        "Phase 13B - Random Forest Optimization"
    ),

    "next_phase": (
        "Phase 14 - Feature Optimization and Ablation"
    ),

    "current_champion": (
        "Tuned Random Forest"
    ),

    "model_file": (
        "phase13B_tuned_random_forest.joblib"
    ),

    "original_rmse": 8.457054,

    "original_accuracy_plus_minus_10": 93.59,

    "tuned_rmse": 8.386059444844367,

    "tuned_accuracy_plus_minus_10": 93.77083219496203,

    "tuned_model_promoted": True,

    "forecast_target": "AQI_t_plus_1",

    "station_count": 10
}

checkpoint_path = os.path.join(
    OUTPUT_PATH,
    "phase13B_checkpoint.json"
)

with open(
    checkpoint_path,
    "w"
) as f:
    json.dump(
        phase13B_checkpoint,
        f,
        indent=4
    )

# Session status
session_status = {
    "completed_phase": "Phase 13B",
    "current_model": "Tuned Random Forest",
    "next_phase": "Phase 14 - Feature Optimization and Ablation",
    "station_count": 10,
    "forecast": "AQI(t) -> AQI(t+1)"
}

status_path = os.path.join(
    OUTPUT_PATH,
    "session_status.json"
)

with open(
    status_path,
    "w"
) as f:
    json.dump(
        session_status,
        f,
        indent=4
    )

print("=" * 60)
print("PHASE 13B CHECKPOINT SAVED")
print("=" * 60)

print(
    "Checkpoint:",
    checkpoint_path,
    "OK"
    if os.path.exists(checkpoint_path)
    else "ERROR"
)

print(
    "Current champion:",
    "Tuned Random Forest"
)

print(
    "Next:",
    "Phase 14 - Feature Optimization and Ablation"
)